# Hell Metrics Notebook

This notebook contains `loader.py` and `compute_genetic_metrics.py` in one place, with setup and run cells.


In [1]:
# Install dependencies in the current kernel environment
%pip install -U pip setuptools wheel
%pip install jedi numpy pandas scipy scikit-learn anndata scanpy pertpy==1.0.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.0/627.0 kB 28.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 117.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 91.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 136.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 90.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 180.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 145.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 MB 148.5 MB/s  0:00:00
  Created wheel for blitzgsea: filename=blitzgsea-1.3.54-py3-none-any.whl size=625752 sha256=c5528dc8050fe65227e5b2c6682188c3eb0747d8dba3db1b14c68b2e1a0fbb9e
  Stored in directory: /root/.cache/pip/wheels/e3/00/84/15131449d6f397371baa95614491906dbd4ca2a93219

In [2]:
import sys, pertpy, scanpy, anndata, sklearn, scipy, pandas, numpy, jedi
print(sys.executable)
print("pertpy", pertpy.__version__)

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:91: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.10.0, so it will not be used.
  warnings.warn(


/usr/bin/python3
pertpy 1.0.6


In [3]:
import sys
import platform
print('Python:', sys.version)
print('Executable:', sys.executable)
print('Machine:', platform.machine())


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Executable: /usr/bin/python3
Machine: x86_64


In [4]:
# ===== loader.py =====
"""Single-file paired AnnData loader.

This module provides a compact, configurable loader for paired real/predicted
AnnData objects with:
- path or in-memory AnnData ingestion
- configurable copy behavior
- schema/column validation
- gene alignment strategies
- optional split-aware condition indexing
"""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal, Optional, Union, Tuple

import numpy as np
import pandas as pd
from scipy import sparse

try:
    import anndata as ad
    import scanpy as sc
except ImportError as exc:  # pragma: no cover
    raise ImportError("anndata and scanpy are required. Install with: pip install anndata scanpy") from exc


# -----------------------------
# Errors
# -----------------------------


class DataLoadError(Exception):
    """Base loader exception."""


class SchemaError(DataLoadError):
    """Raised when required schema elements are missing or invalid."""


class MissingColumnError(SchemaError):
    """Raised when an expected obs column is missing."""


class GeneAlignmentError(DataLoadError):
    """Raised when gene alignment fails."""


class SplitError(DataLoadError):
    """Raised when split configuration is invalid."""


# -----------------------------
# Contracts
# -----------------------------


DataInput = Union[str, Path, ad.AnnData]
ConditionKey = tuple[str, ...]
SplitMap = dict[str, np.ndarray]
MaskMap = dict[ConditionKey, np.ndarray]


@dataclass(frozen=True)
class DatasetSpec:
    """Schema contract for paired datasets."""

    condition_columns: list[str]
    covariate_columns: list[str] = field(default_factory=list)
    split_column: Optional[str] = None
    control_value: Optional[str] = None
    min_cells_per_condition: int = 1
    require_categorical: bool = False


@dataclass(frozen=True)
class LoaderPolicy:
    """Behavior policy for alignment, loading and extraction."""

    gene_alignment: Literal["strict_equal", "intersection", "reference_order"] = "intersection"
    split_policy: Literal["from_obs", "from_external", "none"] = "from_obs"
    dense_mode: Literal["never", "on_extract", "always"] = "on_extract"
    fail_on_warnings: bool = False
    copy_mode: Literal["deep", "none"] = "deep"


@dataclass(frozen=True)
class LoadedAnnData:
    adata: ad.AnnData
    source: str
    backed: bool


@dataclass(frozen=True)
class GeneAlignmentResult:
    ref: ad.AnnData
    pred: ad.AnnData
    common_genes: pd.Index
    dropped_ref: int
    dropped_pred: int


@dataclass(frozen=True)
class ConditionIndex:
    split_to_conditions: dict[str, list[ConditionKey]]
    ref_masks: dict[str, MaskMap]
    pred_masks: dict[str, MaskMap]


@dataclass(frozen=True)
class LoadReport:
    ref_shape: tuple[int, int]
    pred_shape: tuple[int, int]
    n_common_genes: int
    dropped_ref_genes: int
    dropped_pred_genes: int
    split_counts: dict[str, int]
    warnings: list[str]


@dataclass(frozen=True)
class PreparedPair:
    ref: ad.AnnData
    pred: ad.AnnData
    condition_index: ConditionIndex
    report: LoadReport


# -----------------------------
# Components
# -----------------------------


class InputResolver:
    """Resolves path or AnnData input to an AnnData object."""

    def __init__(self, copy_mode: Literal["deep", "none"] = "deep"):
        self.copy_mode = copy_mode

    def resolve(self, data: DataInput, *, backed: Optional[str] = None) -> LoadedAnnData:
        if isinstance(data, (str, Path)):
            path = Path(data)
            if not path.exists():
                raise DataLoadError(f"Input file not found: {path}")
            # read_h5ad ignores backed mode in some versions; sc.read handles backed.
            adata = sc.read(path, backed=backed)
            return LoadedAnnData(adata=adata, source=str(path), backed=backed is not None)

        if not isinstance(data, ad.AnnData):
            raise DataLoadError(f"Unsupported input type: {type(data)}")

        if self.copy_mode == "deep":
            return LoadedAnnData(adata=data.copy(), source="<AnnData>", backed=False)
        return LoadedAnnData(adata=data, source="<AnnData>", backed=False)


class SchemaValidator:
    """Validates obs/var contracts for a pair of AnnData objects."""

    def validate_obs(self, adata: ad.AnnData, spec: DatasetSpec, role: str) -> None:
        required_obs = set(spec.condition_columns) | set(spec.covariate_columns)
        if spec.split_column is not None:
            required_obs.add(spec.split_column)

        missing = [c for c in required_obs if c not in adata.obs.columns]
        if missing:
            raise MissingColumnError(f"Missing required obs columns in {role}: {missing}")

        if spec.require_categorical:
            for col in spec.condition_columns:
                if not pd.api.types.is_categorical_dtype(adata.obs[col]):
                    raise SchemaError(
                        f"Condition column '{col}' in {role} is not categorical. "
                        "Set require_categorical=False or cast it."
                    )

    def validate_var(self, adata: ad.AnnData, role: str) -> None:
        if adata.var_names is None or len(adata.var_names) == 0:
            raise SchemaError(f"{role} has empty var_names")
        if adata.X is None:
            raise SchemaError(f"{role} has no X matrix")

    def validate_pair(self, ref: ad.AnnData, pred: ad.AnnData, spec: DatasetSpec) -> None:
        self.validate_obs(ref, spec, "ref")
        self.validate_obs(pred, spec, "pred")
        self.validate_var(ref, "ref")
        self.validate_var(pred, "pred")


class FeatureAligner:
    """Aligns genes between ref and pred according to policy."""

    def align(self, ref: ad.AnnData, pred: ad.AnnData, policy: LoaderPolicy) -> GeneAlignmentResult:
        ref_genes = pd.Index(ref.var_names.astype(str))
        pred_genes = pd.Index(pred.var_names.astype(str))

        if policy.gene_alignment == "strict_equal":
            if not ref_genes.equals(pred_genes):
                raise GeneAlignmentError("strict_equal policy failed: var_names differ")
            common = ref_genes
            aligned_ref = ref.copy()
            aligned_pred = pred.copy()
        else:
            common = ref_genes.intersection(pred_genes)
            if len(common) == 0:
                raise GeneAlignmentError("No overlapping var_names between ref and pred")

            if policy.gene_alignment == "reference_order":
                # Keep intersection in reference order.
                common = ref_genes[ref_genes.isin(common)]

            ref_idx = ref_genes.get_indexer(common)
            pred_idx = pred_genes.get_indexer(common)
            aligned_ref = ref[:, ref_idx].copy()
            aligned_pred = pred[:, pred_idx].copy()
            aligned_ref.var_names = common
            aligned_pred.var_names = common

        if policy.dense_mode == "always":
            aligned_ref.X = _to_dense(aligned_ref.X)
            aligned_pred.X = _to_dense(aligned_pred.X)

        dropped_ref = len(ref_genes) - len(common)
        dropped_pred = len(pred_genes) - len(common)

        return GeneAlignmentResult(
            ref=aligned_ref,
            pred=aligned_pred,
            common_genes=common,
            dropped_ref=dropped_ref,
            dropped_pred=dropped_pred,
        )


class ConditionIndexer:
    """Builds split-wise condition masks for aligned AnnData pairs."""

    def build(
        self,
        ref: ad.AnnData,
        pred: ad.AnnData,
        spec: DatasetSpec,
        split_map: Optional[SplitMap],
        policy: LoaderPolicy,
    ) -> ConditionIndex:
        split_names = _resolve_splits(ref, spec, split_map, policy)

        split_to_conditions: dict[str, list[ConditionKey]] = {}
        ref_masks: dict[str, MaskMap] = {}
        pred_masks: dict[str, MaskMap] = {}

        for split_name in split_names:
            ref_m = self._build_masks_for_adata(
                adata=ref,
                condition_columns=spec.condition_columns,
                min_cells=spec.min_cells_per_condition,
                split_name=split_name,
                split_column=spec.split_column,
                split_map=split_map,
                is_ref=True,
            )
            pred_m = self._build_masks_for_adata(
                adata=pred,
                condition_columns=spec.condition_columns,
                min_cells=spec.min_cells_per_condition,
                split_name=split_name,
                split_column=spec.split_column,
                split_map=split_map,
                is_ref=False,
            )

            common_keys = sorted(set(ref_m.keys()) & set(pred_m.keys()))
            ref_masks[split_name] = {k: ref_m[k] for k in common_keys}
            pred_masks[split_name] = {k: pred_m[k] for k in common_keys}
            split_to_conditions[split_name] = common_keys

        return ConditionIndex(
            split_to_conditions=split_to_conditions,
            ref_masks=ref_masks,
            pred_masks=pred_masks,
        )

    @staticmethod
    def _build_masks_for_adata(
        adata: ad.AnnData,
        condition_columns: list[str],
        min_cells: int,
        split_name: str,
        split_column: Optional[str],
        split_map: Optional[SplitMap],
        is_ref: bool,
    ) -> MaskMap:
        base_mask = np.ones(adata.n_obs, dtype=bool)

        if split_name != "all":
            if split_map is not None and is_ref:
                idx = split_map.get(split_name)
                if idx is None:
                    raise SplitError(f"split_map missing split '{split_name}'")
                base_mask = np.zeros(adata.n_obs, dtype=bool)
                base_mask[idx] = True
            elif split_column is not None and split_column in adata.obs.columns:
                base_mask &= (adata.obs[split_column].astype(str).values == split_name)

        obs_sub = adata.obs.loc[base_mask, condition_columns].astype(str)
        unique_conditions = obs_sub.drop_duplicates()

        masks: MaskMap = {}
        for _, row in unique_conditions.iterrows():
            key = tuple(str(row[c]) for c in condition_columns)
            mask = base_mask.copy()
            for col in condition_columns:
                mask &= (adata.obs[col].astype(str).values == str(row[col]))
            if int(mask.sum()) >= min_cells:
                masks[key] = mask
        return masks


# -----------------------------
# Facade
# -----------------------------


class PairedAnnDataLoader:
    """Facade for preparing aligned, indexed paired AnnData datasets."""

    def __init__(
        self,
        spec: DatasetSpec,
        policy: LoaderPolicy = LoaderPolicy(),
        validator: Optional[SchemaValidator] = None,
        resolver: Optional[InputResolver] = None,
        aligner: Optional[FeatureAligner] = None,
        indexer: Optional[ConditionIndexer] = None,
    ):
        self.spec = spec
        self.policy = policy
        self.validator = validator or SchemaValidator()
        self.resolver = resolver or InputResolver(copy_mode=policy.copy_mode)
        self.aligner = aligner or FeatureAligner()
        self.indexer = indexer or ConditionIndexer()

    def prepare(
        self,
        ref_data: DataInput,
        pred_data: DataInput,
        *,
        split_map: Optional[SplitMap] = None,
        backed: Optional[str] = None,
    ) -> PreparedPair:
        warnings: list[str] = []

        loaded_ref = self.resolver.resolve(ref_data, backed=backed)
        loaded_pred = self.resolver.resolve(pred_data, backed=backed)

        ref = loaded_ref.adata
        pred = loaded_pred.adata

        self.validator.validate_pair(ref, pred, self.spec)

        alignment = self.aligner.align(ref, pred, self.policy)
        ref_aligned, pred_aligned = alignment.ref, alignment.pred

        if alignment.dropped_ref > 0 or alignment.dropped_pred > 0:
            warnings.append(
                f"Aligned to {len(alignment.common_genes)} common genes "
                f"(dropped ref={alignment.dropped_ref}, pred={alignment.dropped_pred})"
            )

        condition_index = self.indexer.build(
            ref=ref_aligned,
            pred=pred_aligned,
            spec=self.spec,
            split_map=split_map,
            policy=self.policy,
        )

        split_counts = {
            split: len(keys) for split, keys in condition_index.split_to_conditions.items()
        }

        report = LoadReport(
            ref_shape=(int(ref_aligned.n_obs), int(ref_aligned.n_vars)),
            pred_shape=(int(pred_aligned.n_obs), int(pred_aligned.n_vars)),
            n_common_genes=int(len(alignment.common_genes)),
            dropped_ref_genes=int(alignment.dropped_ref),
            dropped_pred_genes=int(alignment.dropped_pred),
            split_counts=split_counts,
            warnings=warnings,
        )

        if self.policy.fail_on_warnings and warnings:
            raise DataLoadError("Warnings encountered with fail_on_warnings=True: " + "; ".join(warnings))

        return PreparedPair(
            ref=ref_aligned,
            pred=pred_aligned,
            condition_index=condition_index,
            report=report,
        )


# -----------------------------
# Utility
# -----------------------------


def _resolve_splits(
    ref: ad.AnnData,
    spec: DatasetSpec,
    split_map: Optional[SplitMap],
    policy: LoaderPolicy,
) -> list[str]:
    if policy.split_policy == "none":
        return ["all"]

    if policy.split_policy == "from_external":
        if split_map is None:
            raise SplitError("split_policy='from_external' requires split_map")
        return list(split_map.keys())

    # from_obs
    if spec.split_column is None:
        return ["all"]
    if spec.split_column not in ref.obs.columns:
        raise SplitError(f"split_column '{spec.split_column}' not found in ref.obs")

    values = ref.obs[spec.split_column].astype(str).unique().tolist()
    return values if values else ["all"]


def _to_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def extract_condition_matrices(
    pair: PreparedPair,
    split: str,
    condition: ConditionKey,
    *,
    dense_mode: Literal["never", "on_extract", "always"] = "on_extract",
) -> Tuple[Union[np.ndarray, sparse.spmatrix], Union[np.ndarray, sparse.spmatrix]]:
    """Extract matched matrices for one split/condition from PreparedPair."""
    if split not in pair.condition_index.ref_masks:
        raise KeyError(f"Unknown split: {split}")

    ref_mask = pair.condition_index.ref_masks[split][condition]
    pred_mask = pair.condition_index.pred_masks[split][condition]

    ref_x = pair.ref.X[ref_mask]
    pred_x = pair.pred.X[pred_mask]

    if dense_mode == "on_extract":
        return _to_dense(ref_x), _to_dense(pred_x)
    return ref_x, pred_x


In [6]:
import types, sys

loader_mod = types.ModuleType("loader")
for name in [
    "DatasetSpec",
    "LoaderPolicy",
    "PairedAnnDataLoader",
    "extract_condition_matrices",
]:
    setattr(loader_mod, name, globals()[name])

sys.modules["loader"] = loader_mod
print("Registered in-memory loader module")

Registered in-memory loader module


In [8]:
# ===== compute_genetic_metrics.py =====
from __future__ import annotations

import argparse
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.spatial.distance import cosine as cosine_distance_fn
from scipy.stats import pearsonr, spearmanr, wasserstein_distance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from loader import (
    DatasetSpec,
    LoaderPolicy,
    PairedAnnDataLoader,
    extract_condition_matrices,
)


VECTOR_METRICS = {
    "euclidean",
    "root_mean_squared_error",
    "mean_absolute_error",
    "pearson_distance",
    "spearman_distance",
    "cosine_distance",
    "r2_distance",
    "mmd",
}

DISTRIBUTION_METRICS = {"edistance", "wasserstein", "sym_kldiv"}


def _to_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def _safe_mean(X: np.ndarray) -> np.ndarray:
    if X.shape[0] == 0:
        raise ValueError("Cannot compute mean on empty matrix")
    return X.mean(axis=0)


def _gaussian_mmd2(X: np.ndarray, Y: np.ndarray, sigma: float | None = None) -> float:
    if X.shape[0] == 0 or Y.shape[0] == 0:
        return np.nan

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)

    if sigma is None:
        joint = np.vstack([X, Y])
        if joint.shape[0] > 2000:
            idx = np.random.RandomState(42).choice(joint.shape[0], 2000, replace=False)
            joint = joint[idx]
        dists = np.sum((joint[:, None, :] - joint[None, :, :]) ** 2, axis=2)
        med = np.median(dists[dists > 0]) if np.any(dists > 0) else 1.0
        sigma = np.sqrt(max(med, 1e-12))

    gamma = 1.0 / (2.0 * sigma * sigma)

    XX = np.exp(-gamma * np.sum((X[:, None, :] - X[None, :, :]) ** 2, axis=2))
    YY = np.exp(-gamma * np.sum((Y[:, None, :] - Y[None, :, :]) ** 2, axis=2))
    XY = np.exp(-gamma * np.sum((X[:, None, :] - Y[None, :, :]) ** 2, axis=2))

    return float(XX.mean() + YY.mean() - 2.0 * XY.mean())


def _vector_metric(metric: str, real_x: np.ndarray, pred_x: np.ndarray) -> float:
    r = _safe_mean(real_x)
    p = _safe_mean(pred_x)

    if metric == "euclidean":
        return float(np.linalg.norm(p - r))
    if metric == "root_mean_squared_error":
        return float(np.sqrt(mean_squared_error(r, p)))
    if metric == "mean_absolute_error":
        return float(mean_absolute_error(r, p))
    if metric == "pearson_distance":
        corr = pearsonr(r, p)[0]
        if np.isnan(corr):
            return np.nan
        return float(1.0 - corr)
    if metric == "spearman_distance":
        corr = spearmanr(r, p)[0]
        if np.isnan(corr):
            return np.nan
        return float(1.0 - corr)
    if metric == "cosine_distance":
        return float(cosine_distance_fn(r, p))
    if metric == "r2_distance":
        return float(1.0 - r2_score(r, p))
    if metric == "mmd":
        return _gaussian_mmd2(pred_x, real_x)

    raise ValueError(f"Unsupported vector metric: {metric}")


def _distribution_metric(metric: str, real_x: np.ndarray, pred_x: np.ndarray) -> float:
    if metric == "wasserstein":
        # 1D approximation by averaging feature-wise Wasserstein distances.
        vals = [wasserstein_distance(real_x[:, i], pred_x[:, i]) for i in range(real_x.shape[1])]
        return float(np.mean(vals))

    # Use pertpy for remaining distribution metrics.
    try:
        import anndata as ad
        import pertpy as pt
    except Exception as exc:  # pragma: no cover
        raise RuntimeError(
            "pertpy is required for distribution metrics (edistance/sym_kldiv). "
            "Install requirements from hell/requirements.txt"
        ) from exc

    obs_real = pd.DataFrame({"Expcategory": ["stimulated"] * real_x.shape[0]})
    obs_pred = pd.DataFrame({"Expcategory": ["imputed"] * pred_x.shape[0]})

    merged = ad.concat(
        [ad.AnnData(X=real_x, obs=obs_real), ad.AnnData(X=pred_x, obs=obs_pred)],
        join="inner",
    )
    merged.layers["X"] = merged.X

    distance = pt.tools.Distance(metric=metric, layer_key="X")
    pairwise_df = distance.onesided_distances(
        merged,
        groupby="Expcategory",
        selected_group="imputed",
        groups=["stimulated"],
    )
    val = float(pairwise_df["stimulated"])
    if metric == "sym_kldiv":
        val = float(np.log2(val + 1.0))
    return val


def compute_metrics(
    real_path: Path,
    pred_path: Path,
    condition_column: str,
    metrics: List[str],
    min_cells_per_condition: int,
) -> pd.DataFrame:
    spec = DatasetSpec(
        condition_columns=[condition_column],
        split_column=None,
        min_cells_per_condition=min_cells_per_condition,
    )
    policy = LoaderPolicy(
        gene_alignment="intersection",
        split_policy="none",
        dense_mode="on_extract",
        copy_mode="deep",
    )

    pair = PairedAnnDataLoader(spec=spec, policy=policy).prepare(real_path, pred_path)

    split = "all"
    conditions = pair.condition_index.split_to_conditions[split]

    rows: List[Dict] = []
    for condition in conditions:
        real_x, pred_x = extract_condition_matrices(pair, split=split, condition=condition, dense_mode="on_extract")
        real_x = _to_dense(real_x)
        pred_x = _to_dense(pred_x)

        if real_x.shape[0] == 0 or pred_x.shape[0] == 0:
            continue

        for metric in metrics:
            if metric in VECTOR_METRICS:
                score = _vector_metric(metric, real_x, pred_x)
            elif metric in DISTRIBUTION_METRICS:
                score = _distribution_metric(metric, real_x, pred_x)
            else:
                raise ValueError(f"Unsupported metric: {metric}")

            rows.append(
                {
                    "condition": condition[0],
                    "metric": metric,
                    "score": score,
                    "n_real": int(real_x.shape[0]),
                    "n_pred": int(pred_x.shape[0]),
                    "n_genes": int(real_x.shape[1]),
                }
            )

    return pd.DataFrame(rows)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Compute perturbation generalization metrics on paired h5ad files.")
    parser.add_argument("--real", required=True, help="Path to real/reference .h5ad")
    parser.add_argument("--pred", required=True, help="Path to predicted .h5ad")
    parser.add_argument(
        "--condition-column",
        default="perturbation",
        help="obs column used to define perturbation condition (default: perturbation)",
    )
    parser.add_argument(
        "--metrics",
        nargs="+",
        default=[
            "euclidean",
            "root_mean_squared_error",
            "mean_absolute_error",
            "pearson_distance",
            "spearman_distance",
            "cosine_distance",
            "r2_distance",
            "mmd",
        ],
        help="Metrics to compute",
    )
    parser.add_argument("--min-cells", type=int, default=1, help="Minimum cells per condition per dataset")
    parser.add_argument("--out", default="hell/metrics_output.csv", help="Output CSV path")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    df = compute_metrics(
        real_path=Path(args.real),
        pred_path=Path(args.pred),
        condition_column=args.condition_column,
        metrics=args.metrics,
        min_cells_per_condition=args.min_cells,
    )

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out, index=False)

    print(f"Wrote {len(df)} rows to {out}")
    if not df.empty:
        summary = df.groupby("metric", as_index=False)["score"].mean().rename(columns={"score": "mean_score"})
        print("\nMean score per metric:")
        print(summary.to_string(index=False))


# if __name__ == "__main__":
#     main()


In [15]:
# Run metrics (start with vector metrics; add distribution metrics later if needed)
from pathlib import Path

real_path = Path('/content/real_renamed.h5ad')
pred_path = Path('/content/pred_renamed.h5ad')
out_path = Path('/content/fold0_metrics_vector_no_mmd.csv')



metrics = [
    'euclidean',
    'root_mean_squared_error',
    'mean_absolute_error',
    'pearson_distance',
    'spearman_distance',
    'cosine_distance',
    'r2_distance',
]

df = compute_metrics(
    real_path=real_path,
    pred_path=pred_path,
    condition_column='condition',
    metrics=metrics,
    min_cells_per_condition=1,
)

# out_path.parent.mkdir(parents=True, exist_ok=True)
# df.to_csv(out_path, index=False)
# print(f'Wrote {len(df)} rows to {out_path}')
# df.head()

# wide format (one row per condition)
id_cols = ['condition', 'n_real', 'n_pred', 'n_genes']
df_wide = (
    df.pivot_table(index=id_cols, columns='metric', values='score', aggfunc='first')
      .reset_index()
)
df_wide.columns.name = None
metric_cols = sorted([c for c in df_wide.columns if c not in id_cols])
df_wide = df_wide[id_cols + metric_cols]

out_wide = Path('/content/fold0_metrics_vector_no_mmd_wide.csv')
df_wide.to_csv(out_wide, index=False)
print(f'Wrote {len(df_wide)} wide rows to {out_wide}')

df_wide.head()


Wrote 38 wide rows to /content/fold0_metrics_vector_no_mmd_wide.csv


,condition,n_real,n_pred,n_genes,cosine_distance,euclidean,mean_absolute_error,pearson_distance,r2_distance,root_mean_squared_error,spearman_distance
0,AHR+FEV,239,128,1000,0.011489,2.611066,0.031689,0.012593,0.025508,0.082569,0.229822
1,BPGM+SAMD1,233,128,1000,0.004359,1.592740,0.021719,0.004482,0.009848,0.050367,0.191881
2,CBL+UBASH3A,48,128,1000,0.005011,1.706048,0.021317,0.005418,0.011054,0.053950,0.251971
3,CBL+UBASH3B,311,128,1000,0.003893,1.574467,0.018797,0.004057,0.009074,0.049789,0.226046
4,CDKN1B+CDKN1A,98,128,1000,0.003975,1.504918,0.021551,0.004089,0.008863,0.047590,0.195465


In [14]:
# out_path1 = Path("/content/fold0_more.csv")

# metrics1 = [
#     "wasserstein",
#     "edistance",
#     "sym_kldiv",
# ]

# df1 = compute_metrics(
#     real_path=real_path,
#     pred_path=pred_path,
#     condition_column="condition",
#     metrics=metrics1,
#     min_cells_per_condition=1,
# )

# out_path1.parent.mkdir(parents=True, exist_ok=True)
# df1.to_csv(out_path1, index=False)
# print(f"Wrote {len(df1)} rows to {out_path1}")
# df1.head()

Output()

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Output()

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Output()

...

Output()

Output()

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Wrote 114 rows to /content/fold0_more.csv


,condition,metric,score,n_real,n_pred,n_genes
0,AHR+FEV,wasserstein,0.036186,239,128,1000
1,AHR+FEV,edistance,0.425262,239,128,1000
2,AHR+FEV,sym_kldiv,41.593414,239,128,1000
3,BPGM+SAMD1,wasserstein,0.033480,233,128,1000
4,BPGM+SAMD1,edistance,0.381810,233,128,1000


In [11]:
import os
print(os.path.exists("/content/real_renamed.h5ad"))

True
